<a href="https://colab.research.google.com/github/SerraGuney/Optimized-Transfer-Learning-for-Multi-Class-Plant-Disease-Classification/blob/main/Optimized_Transfer_Learning_for_Multi_Class_Plant_Disease_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Veri setini colab dosyasına indirme

# Bu çalışmanın amacı, PlantVillage veri setindeki çok sınıflı yaprak görüntülerini kullanarak, 38 farklı sınıfa ait bitki hastalıklarını derin öğrenme ve transfer learning yöntemleri ile doğru şekilde sınıflandırmaktır.

Mobilenet ile trasfer learning gerçekleştircez.çünkü daha az karmaşık ve kullanması daha kolaydır.

**1. Veri Yükleme ve data augmentation**

In [ ]:
from google.colab import userdata
import os

# 1. Kaggle API anahtarlarını yükle
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

# 2. Veri indir ve ayıkla
if not os.path.exists('/content/data/color'):
    print(" İndirme başlıyor...")
    !kaggle datasets download -d abdallahalidev/plantvillage-dataset

    print(" Zip açılıyor...")
    !unzip -q plantvillage-dataset.zip -d /content/data

    # EKRAN GÖRÜNTÜSÜNDEKİ GERÇEK YOL: /content/data/plantvillage dataset/color
    # Not: Klasör isminde boşluk olduğu için tırnak işareti (" ") kullandık.
    print(" Klasörler taşınıyor...")
    !mv "/content/data/plantvillage dataset/color" /content/data/

    # Gereksizleri temizle
    !rm -rf "/content/data/plantvillage dataset"
    !rm plantvillage-dataset.zip

    print(" İşlem tamam! /content/data/color klasörü hazır.")
else:
    print(" Veriler zaten mevcut.")

In [ ]:
import os

# Data klasörünün içinde gerçekten ne var?
print("Data içindekiler:", os.listdir('/content/data'))

# Eğer boşsa, bir de ana dizine bakalım, belki yanlış yere açılmıştır
print("Ana dizin içindekiler:", os.listdir('/content'))


** Kütüphane Bilgilendirmeleri**
**torch:** Temel kütüphanedir. GPU destekli çok boyutlu dizileri (tensor) ve matematiksel işlemleri yönetir.

**torch.nn:** Sinir ağı katmanlarını (Convolution, Linear, Dropout vb.) ve kayıp fonksiyonlarını (Loss functions) oluşturmak için kullanılır.

**torch.optim:** Modelin parametrelerini (ağırlıklarını) güncellemek için kullanılan algoritmaları (Adam, SGD vb.) barındırır.

**torchvision.transforms:** Görüntüleri Tensor formatına çevirme, yeniden boyutlandırma ve normalizasyon gibi ön işleme işlemlerini yapar.

**torchvision.datasets:** Dosya sistemindeki görüntüleri otomatik olarak etiketleyip veri kümesi (dataset) nesnesine dönüştürür.

**torchvision.models:** Önceden eğitilmiş (Pre-trained) mimarileri (ResNet50, MobileNetV2 vb.) projeye dahil eder.

**DataLoader: **Veri kümesini bellek yönetimine uygun şekilde küçük gruplara (batch) böler ve karıştırarak (shuffle) modele sunar.

**tqdm:** Döngülerin (eğitim ve test aşamaları) ilerleme durumunu ve süresini konsolda görselleştirir.

**matplotlib.pyplot:**Eğitim sırasındaki hata ve doğruluk oranlarını grafik olarak raporlamak için kullanılır.

**seaborn:** Matplotlib tabanlıdır; karmaşıklık matrisi (confusion matrix) gibi istatistiksel verileri daha okunabilir şekilde görselleştirir.

**sklearn.metrics:** Modelin performansını ölçmek için kesinlik (precision), duyarlılık (recall) ve F1 skoru gibi metrikleri hesaplar.


In [ ]:
import torch
import torch.nn as nn # sinir ağı katmanları için
import torch.optim as optim # sinir ağındaki optimizasyon algoritmaları için
import torchvision.transforms as transforms # görüntü dönüşümleri için (data augmentataion)
import torchvision.datasets as datasets # görüntü veri kümeleri için
import torchvision.models as models # önceden eğitilmiş modelleri yüklemek için(mobilenet,...)
from torch.utils.data import DataLoader #batchler oluşturur.
from tqdm import tqdm # eğitim sürecini izlemek için kullancağımız bir ilerleme cubugu
import matplotlib.pyplot as plt # görselleştirme için
import seaborn as sns #confuxion matris için
from sklearn.metrics import confusion_matrix, classification_report #confuxion matris için


**veri dönüşümü**
1.   klasik dönüşümler:normalizasyon, tensor dönüşümü
2.   mobilnet e uygun giris boyutunun ayarlanması
3.   data augmentation





In [ ]:
transform_train=transforms.Compose([
    transforms.Resize((224,224)), #mobile net input size yazmamız lazım.
    transforms.RandomHorizontalFlip(), # görüntüleri yatay çevirerek veri arttırımı.
    transforms.RandomRotation(10), # görüntüyü rastgele 10 dereceye kadar döndürür.
    transforms.ColorJitter(brightness=0.2, contrast=0.2,saturation=0.2, hue=0.1) , # aynı görüntünün renk varyansolarını sağlar.
    transforms.ToTensor(),#goruntüleri tensore(C,H,W) ve piksel değerlerini 0–1 aralığına çevir.
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5)) #piksel değerlerini normalize eder.
])
#test veri setinde augmentation uygulanmaz(horizontal,rotation,..)
transform_test=transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

veri setini split etmek.

Toplam resim sayısı ~54.305 (38013 + 8145 + 8147 ≈ 54.305).

Eğitim: %70 → 38.013

Doğrulama: %15 → 8.145

Test: %15 → 8.147

In [ ]:
import torch
from torchvision import datasets
from torch.utils.data import DataLoader, random_split, Dataset

# 1. Ana veri kümesini oku (Transformsız)
full_dataset = datasets.ImageFolder(root='/content/data/color')

# 2. Boyutları hesapla
total_count = len(full_dataset)
train_count = int(0.7 * total_count)
val_count = int(0.15 * total_count)
test_count = total_count - train_count - val_count

# 3. Sadece indeksleri böl
train_indices, val_indices, test_indices = random_split(
    range(total_count),
    [train_count, val_count, test_count],
    generator=torch.Generator().manual_seed(42)
)

# 4. Kendi transformunu taşıyan özel bir sınıf (Wrapper)
class ApplyTransform(Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform

    def __getitem__(self, index):
        x, y = self.subset[index]
        if self.transform:
            x = self.transform(x)
        return x, y

    def __len__(self):
        return len(self.subset)

# 5. Parçaları oluştur ve transformları ata
# Burada full_dataset'i indekslere göre böldükten sonra transformları uyguluyoruz
train_dataset = ApplyTransform(torch.utils.data.Subset(full_dataset, train_indices), transform=transform_train)
val_dataset = ApplyTransform(torch.utils.data.Subset(full_dataset, val_indices), transform=transform_test)
test_dataset = ApplyTransform(torch.utils.data.Subset(full_dataset, test_indices), transform=transform_test)

print(f" Bölme bitti ve transformlar ayrıştırıldı!")
print(f"Eğitim: {len(train_dataset)} | Doğrulama: {len(val_dataset)} | Test: {len(test_dataset)}")

Veri seti hakkında bilgi edinmek için rastgele 5 örnek seçip görselleştirme.


In [ ]:
import torch
import matplotlib.pyplot as plt
import textwrap

# 1. Sınıf isimlerini al
class_names = full_dataset.classes

# 2. Rastgele 5 indeks seç
indices = torch.randint(len(train_dataset), (5,))
samples = [train_dataset[i] for i in indices]

# 3. Görselleştirme
fig, axes = plt.subplots(1, 5, figsize=(20, 6))

for i, (image, label) in enumerate(samples):
    # Tensor'dan (C, H, W) -> Numpy'a (H, W, C) dönüşümü
    image = image.numpy().transpose((1, 2, 0))

    # Normalizasyonu geri al (Normalize işlemi: (x-0.5)/0.5 yapmıştık, şimdi geri çeviriyoruz)
    image = (image * 0.5) + 0.5
    image = image.clip(0, 1) # Piksel değerlerinin 0-1 arasında olduğundan emin ol

    axes[i].imshow(image)

    # İsim Formatlama: Alt tireleri boşluğa çevir ve 15 karakterde bir böl
    raw_name = class_names[label].replace("___", " ").replace("_", " ")
    wrapped_name = "\n".join(textwrap.wrap(raw_name, width=15))

    axes[i].set_title(wrapped_name, fontsize=12, pad=10, fontweight='semibold')
    axes[i].axis('off')

plt.subplots_adjust(wspace=0.3)
plt.show()

In [ ]:
#data loader

# shuffle=True:veri setindeki resimlerin sırasını rastgele karıştırır.
train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True)

# Doğrulama ve Test için Shuffle=False (Sadece ölçüm yapıyoruz, karıştırmaya gerek yok)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

**2. Transfer learning tanımlama ve fine tuning ve model kaydetme**

In [ ]:
# mobilenet V2 yükleme


# sınıflandırıcı katmanı ekleme


# kayıp fonksiyonu ve optimizer tanımlama


# model training


# save model

**3. Test ve değerlendirme**